In [ ]:
import numpy as np
import pandas as pd
import time
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
# ============================================================
# HARDWARE MONITORING
# ============================================================

!pip install -q psutil nvidia-ml-py


import os
import time
import threading
import numpy as np
import pandas as pd
import psutil


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 1.7 MB/s eta 0:00:00


In [ ]:
# ------------------------------------------------------------
# NVIDIA NVML
# ------------------------------------------------------------

try:
    import pynvml

    pynvml.nvmlInit()
    NVML_AVAILABLE = True

except Exception:
    NVML_AVAILABLE = False

In [ ]:
# ------------------------------------------------------------
# Hardware Monitor
# ------------------------------------------------------------

class HardwareMonitor:

    def __init__(self, interval=0.2, gpu_index=0):

        self.interval = interval
        self.gpu_index = gpu_index

        self.running = False
        self.thread = None

        # CPU
        self.cpu_util_samples = []
        self.ram_samples = []

        # GPU
        self.gpu_util_samples = []
        self.gpu_mem_util_samples = []
        self.vram_samples = []
        self.gpu_power_samples = []
        self.gpu_temp_samples = []

        self.timestamps = []

        self.process = psutil.Process(os.getpid())

        self.gpu_handle = None

        if NVML_AVAILABLE:

            try:
                self.gpu_handle = (
                    pynvml.nvmlDeviceGetHandleByIndex(
                        gpu_index
                    )
                )

            except Exception:
                self.gpu_handle = None

    # --------------------------------------------------------
    # Start
    # --------------------------------------------------------
    def start(self):

        # Initialize process CPU counter
        self.process.cpu_percent(None)

        self.running = True

        self.thread = threading.Thread(
            target=self._monitor,
            daemon=True
        )

        self.thread.start()

    # --------------------------------------------------------
    # Monitoring loop
    # --------------------------------------------------------
    def _monitor(self):

        logical_cpus = psutil.cpu_count(
            logical=True
        )

        while self.running:

            timestamp = time.perf_counter()
            # =================================================
            # CPU UTILIZATION
            # =================================================
            process_cpu = (
                self.process.cpu_percent(
                    interval=None
                )
            )

            # Normalize process CPU usage to
            # percentage of total logical CPU capacity.
            if logical_cpus:
                process_cpu_normalized = (
                    process_cpu / logical_cpus
                )
            else:
                process_cpu_normalized = process_cpu
            self.cpu_util_samples.append(
                process_cpu_normalized
            )

            # =================================================
            # RAM
            # =================================================

            ram_mb = (
                self.process.memory_info().rss
                / (1024 ** 2)
            )

            self.ram_samples.append(
                ram_mb
            )

            # =================================================
            # GPU
            # =================================================

            if self.gpu_handle is not None:

                try:

                    utilization = (
                        pynvml.nvmlDeviceGetUtilizationRates(
                            self.gpu_handle
                        )
                    )

                    memory = (
                        pynvml.nvmlDeviceGetMemoryInfo(
                            self.gpu_handle
                        )
                    )

                    power = (
                        pynvml.nvmlDeviceGetPowerUsage(
                            self.gpu_handle
                        ) / 1000.0
                    )

                    temperature = (
                        pynvml.nvmlDeviceGetTemperature(
                            self.gpu_handle,
                            pynvml.NVML_TEMPERATURE_GPU
                        )
                    )

                    # GPU compute utilization
                    self.gpu_util_samples.append(
                        float(utilization.gpu)
                    )

                    # GPU memory-controller utilization
                    self.gpu_mem_util_samples.append(
                        float(utilization.memory)
                    )

                    # VRAM used
                    self.vram_samples.append(
                        memory.used / (1024 ** 2)
                    )

                    # Power in Watts
                    self.gpu_power_samples.append(
                        power
                    )

                    # Temperature
                    self.gpu_temp_samples.append(
                        float(temperature)
                    )

                except Exception:
                    pass

            self.timestamps.append(
                timestamp
            )

            time.sleep(
                self.interval
            )

    # --------------------------------------------------------
    # Stop
    # --------------------------------------------------------

    def stop(self):

        self.running = False

        if self.thread is not None:
            self.thread.join()

    # --------------------------------------------------------
    # Results
    # --------------------------------------------------------

    def get_results(self):

        result = {}

        # =====================================================
        # CPU
        # =====================================================

        result["avg_cpu_util_percent"] = (
            np.mean(
                self.cpu_util_samples
            )
            if self.cpu_util_samples
            else np.nan
        )

        result["peak_cpu_util_percent"] = (
            np.max(
                self.cpu_util_samples
            )
            if self.cpu_util_samples
            else np.nan
        )

        result["avg_ram_mb"] = (
            np.mean(
                self.ram_samples
            )
            if self.ram_samples
            else np.nan
        )

        result["peak_ram_mb"] = (
            np.max(
                self.ram_samples
            )
            if self.ram_samples
            else np.nan
        )

        # =====================================================
        # GPU
        # =====================================================

        if self.gpu_util_samples:

            result["avg_gpu_util_percent"] = np.mean(
                self.gpu_util_samples
            )

            result["peak_gpu_util_percent"] = np.max(
                self.gpu_util_samples
            )

            result["avg_gpu_memory_util_percent"] = np.mean(
                self.gpu_mem_util_samples
            )

            result["peak_gpu_memory_util_percent"] = np.max(
                self.gpu_mem_util_samples
            )

            result["avg_vram_mb"] = np.mean(
                self.vram_samples
            )

            result["peak_vram_mb"] = np.max(
                self.vram_samples
            )

            result["avg_gpu_power_w"] = np.mean(
                self.gpu_power_samples
            )

            result["peak_gpu_power_w"] = np.max(
                self.gpu_power_samples
            )

            result["avg_gpu_temperature_c"] = np.mean(
                self.gpu_temp_samples
            )

            result["peak_gpu_temperature_c"] = np.max(
                self.gpu_temp_samples
            )

            # =================================================
            # Energy
            # E = integral(P dt)
            # Trapezoidal approximation
            # =================================================

            energy_joules = 0.0

            n = min(
                len(self.gpu_power_samples),
                len(self.timestamps)
            )

            for i in range(1, n):

                dt = (
                    self.timestamps[i]
                    - self.timestamps[i - 1]
                )

                avg_power = (
                    self.gpu_power_samples[i]
                    + self.gpu_power_samples[i - 1]
                ) / 2.0

                energy_joules += (
                    avg_power * dt
                )

            result["gpu_energy_joules"] = (
                energy_joules
            )

        else:

            result["avg_gpu_util_percent"] = np.nan
            result["peak_gpu_util_percent"] = np.nan

            result["avg_gpu_memory_util_percent"] = np.nan
            result["peak_gpu_memory_util_percent"] = np.nan

            result["avg_vram_mb"] = np.nan
            result["peak_vram_mb"] = np.nan

            result["avg_gpu_power_w"] = np.nan
            result["peak_gpu_power_w"] = np.nan

            result["avg_gpu_temperature_c"] = np.nan
            result["peak_gpu_temperature_c"] = np.nan

            result["gpu_energy_joules"] = np.nan

        return result

In [ ]:
# ------------------------------------------------------------
# Save 5-run results
# ------------------------------------------------------------

def save_five_run_results(
    run_results,
    filename,
    algorithm,
    device
):

    rows = []

    for result in run_results:

        row = result.copy()

        row["Algorithm"] = algorithm
        row["Device"] = device

        rows.append(row)

    runs_df = pd.DataFrame(rows)

    # --------------------------------------------------------
    # Calculate average
    # --------------------------------------------------------

    numeric_columns = [
        c for c in runs_df.columns
        if c not in ["run"]
        and pd.api.types.is_numeric_dtype(
            runs_df[c]
        )
    ]

    average_row = {
        "run": "AVERAGE",
        "Algorithm": algorithm,
        "Device": device
    }

    for column in numeric_columns:

        average_row[column] = (
            runs_df[column].mean()
        )

    # --------------------------------------------------------
    # Calculate standard deviation
    # --------------------------------------------------------

    std_row = {
        "run": "STD",
        "Algorithm": algorithm,
        "Device": device
    }

    for column in numeric_columns:

        std_row[column] = (
            runs_df[column].std()
        )

    final_df = pd.concat(
        [
            runs_df,
            pd.DataFrame([
                average_row,
                std_row
            ])
        ],
        ignore_index=True
    )

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    final_df.to_csv(
        filename,
        index=False
    )

    print("\n" + "=" * 80)
    print(f"{algorithm} - {device}")
    print("=" * 80)

    display(
        final_df.round(3)
    )

    print(
        f"\nSaved result file:\n{filename}"
    )

    return final_df

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

RESULT_DIR = (
    "/content/drive/MyDrive/"
    "LLORMA_Project/hardware_results"
)

os.makedirs(
    RESULT_DIR,
    exist_ok=True
)

print(
    "Results directory:",
    RESULT_DIR
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results directory: /content/drive/MyDrive/LLORMA_Project/hardware_results


In [ ]:
df = pd.read_csv('ratings.csv')
print(df.columns.tolist())
print(df.shape)
df.head()

['userId', 'movieId', 'rating', 'timestamp']
(100836, 4)


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [ ]:
col_map = {}
for c in df.columns:
    lc = c.lower().replace('_', '')
    if 'user' in lc: col_map[c] = 'user'
    elif 'movie' in lc or 'item' in lc: col_map[c] = 'item'
    elif 'rating' in lc: col_map[c] = 'rating'
df = df.rename(columns=col_map)

df = df[['user', 'item', 'rating']].dropna()

user_ids = df['user'].astype('category').cat.codes.values
item_ids = df['item'].astype('category').cat.codes.values
ratings  = df['rating'].astype(np.float32).values

n_users = user_ids.max() + 1
n_items = item_ids.max() + 1
print(f"Users: {n_users}, Items: {n_items}, Ratings: {len(ratings)}")

np.random.seed(RANDOM_SEED)
idx = np.random.permutation(len(ratings))
split = int(0.8 * len(ratings))
train_idx, test_idx = idx[:split], idx[split:]

train_u, train_i, train_r = user_ids[train_idx], item_ids[train_idx], ratings[train_idx]
test_u,  test_i,  test_r  = user_ids[test_idx],  item_ids[test_idx],  ratings[test_idx]

global_mean = train_r.mean()
print(f"Train: {len(train_r)}, Test: {len(test_r)}, Global mean: {global_mean:.3f}")

Users: 610, Items: 9724, Ratings: 100836
Train: 80668, Test: 20168, Global mean: 3.503


In [ ]:
train_mat = csr_matrix((train_r - global_mean, (train_u, train_i)), shape=(n_users, n_items))

EMBED_DIM = 5  # small rank, just used for distance/kernel, not final predictions
U_svd, S, Vt = svds(train_mat.asfptype(), k=EMBED_DIM)
U_svd, S, Vt = U_svd[:, ::-1], S[::-1], Vt[::-1, :]  # svds returns ascending order

U_embed = U_svd * np.sqrt(S)     # (n_users, EMBED_DIM)
V_embed = (Vt.T) * np.sqrt(S)    # (n_items, EMBED_DIM)
print("Global embedding shapes:", U_embed.shape, V_embed.shape)

Global embedding shapes: (610, 5) (9724, 5)


In [ ]:
def epanechnikov_kernel(dist, bandwidth):
    """0.75 * (1 - (d/h)^2) for d < h, else 0"""
    ratio = dist / bandwidth
    return np.where(ratio < 1.0, 0.75 * (1.0 - ratio ** 2), 0.0)

In [ ]:
def train_local_model(train_u, train_i, train_r, weights, mask,
                       n_users, n_items, rank=5, n_epochs=15,
                       lr=0.01, reg=0.05, seed=0, batch_size=512):
    rng = np.random.RandomState(seed)
    P = rng.normal(0, 0.1, (n_users, rank)).astype(np.float32)
    Q = rng.normal(0, 0.1, (n_items, rank)).astype(np.float32)

    u_sub, i_sub, r_sub, w_sub = train_u[mask], train_i[mask], train_r[mask], weights[mask]
    n_samples = len(u_sub)
    if n_samples == 0:
        return P, Q

    for epoch in range(n_epochs):
        perm = rng.permutation(n_samples)
        u_sub, i_sub, r_sub, w_sub = u_sub[perm], i_sub[perm], r_sub[perm], w_sub[perm]

        for start in range(0, n_samples, batch_size):
            end = start + batch_size
            bu, bi, br, bw = u_sub[start:end], i_sub[start:end], r_sub[start:end], w_sub[start:end]

            pu, qi = P[bu], Q[bi]
            pred = np.sum(pu * qi, axis=1)
            err = (br - pred) * bw

            grad_p = -2 * err[:, None] * qi + 2 * reg * pu
            grad_q = -2 * err[:, None] * pu + 2 * reg * qi

            np.add.at(P, bu, -lr * grad_p)
            np.add.at(Q, bi, -lr * grad_q)

    return P, Q

In [ ]:
def predict_llorma(local_models, users, items, global_mean):
    preds = np.zeros(len(users), dtype=np.float64)
    weight_sums = np.zeros(len(users), dtype=np.float64)

    for m in local_models:
        w = m['wu'][users] * m['wi'][items]
        active = w > 1e-6
        if not np.any(active):
            continue
        local_pred = np.sum(m['P'][users[active]] * m['Q'][items[active]], axis=1)
        preds[active] += w[active] * local_pred
        weight_sums[active] += w[active]

    covered = weight_sums > 1e-6
    preds[covered] /= weight_sums[covered]
    preds[~covered] = global_mean

    return np.clip(preds, 0.5, 5.0)

In [ ]:
def train_llorma(train_u, train_i, train_r, U_embed, V_embed,
                  n_users, n_items, n_anchors=20, rank=5,
                  bandwidth=0.8, n_epochs=15, lr=0.01, reg=0.05):
    rng = np.random.RandomState(RANDOM_SEED)
    anchor_positions = rng.choice(len(train_u), size=n_anchors, replace=False)
    local_models = []

    for t, pos in enumerate(anchor_positions):
        au, ai = train_u[pos], train_i[pos]

        du = np.linalg.norm(U_embed - U_embed[au], axis=1)
        di = np.linalg.norm(V_embed - V_embed[ai], axis=1)

        wu = epanechnikov_kernel(du, bandwidth)
        wi = epanechnikov_kernel(di, bandwidth)

        sample_w = wu[train_u] * wi[train_i]
        mask = sample_w > 1e-6
        if mask.sum() < 20:
            continue

        P, Q = train_local_model(train_u, train_i, train_r, sample_w, mask,
                                  n_users, n_items, rank=rank, n_epochs=n_epochs,
                                  lr=lr, reg=reg, seed=t)

        local_models.append({'au': au, 'ai': ai, 'P': P, 'Q': Q, 'wu': wu, 'wi': wi})
        print(f"Anchor {t+1}/{n_anchors} trained | neighbors used: {mask.sum()}")

    return local_models

In [ ]:
# ============================================================
# LLORMA CPU - 5 RUN HARDWARE EXPERIMENT
# ============================================================

N_RUNS = 5

N_ANCHORS = 20
RANK = 5
BANDWIDTH = 0.8
N_EPOCHS = 15
LR = 0.01
REG = 0.05


llorma_cpu_runs = []


for run in range(1, N_RUNS + 1):

    print(
        f"\n{'=' * 25}"
        f" LLORMA CPU RUN {run}/5 "
        f"{'=' * 25}"
    )

    # --------------------------------------------------------
    # Start monitoring
    # --------------------------------------------------------

    monitor = HardwareMonitor(
        interval=0.2
    )

    monitor.start()

    start_time = time.perf_counter()

    # --------------------------------------------------------
    # EXACT EXISTING LLORMA TRAINING
    # --------------------------------------------------------

    local_models_cpu = train_llorma(
        train_u,
        train_i,
        train_r,

        U_embed,
        V_embed,

        n_users,
        n_items,

        n_anchors=N_ANCHORS,
        rank=RANK,
        bandwidth=BANDWIDTH,
        n_epochs=N_EPOCHS,
        lr=LR,
        reg=REG
    )

    # --------------------------------------------------------
    # Timing
    # --------------------------------------------------------

    elapsed = (
        time.perf_counter()
        - start_time
    )

    monitor.stop()

    result = monitor.get_results()

    result["run"] = run
    result["training_time_s"] = elapsed

    llorma_cpu_runs.append(
        result
    )

    print(
        f"Training time: {elapsed:.4f} s"
    )


# ============================================================
# SAVE
# ============================================================

llorma_cpu_file = os.path.join(
    RESULT_DIR,
    "LLORMA_CPU_hardware_results.csv"
)

llorma_cpu_results_df = (
    save_five_run_results(
        llorma_cpu_runs,
        llorma_cpu_file,
        "LLORMA",
        "CPU"
    )
)


========================= LLORMA CPU RUN 1/5 =========================
Anchor 1/20 trained | neighbors used: 53609
Anchor 2/20 trained | neighbors used: 16405
Anchor 3/20 trained | neighbors used: 36613
Anchor 4/20 trained | neighbors used: 50717
Anchor 5/20 trained | neighbors used: 4091
Anchor 6/20 trained | neighbors used: 52875
Anchor 7/20 trained | neighbors used: 17986
Anchor 8/20 trained | neighbors used: 6688
Anchor 9/20 trained | neighbors used: 55428
Anchor 10/20 trained | neighbors used: 2468
Anchor 11/20 trained | neighbors used: 5678
Anchor 12/20 trained | neighbors used: 21140
Anchor 13/20 trained | neighbors used: 50084
Anchor 14/20 trained | neighbors used: 55210
Anchor 15/20 trained | neighbors used: 43893
Anchor 16/20 trained | neighbors used: 50481
Anchor 17/20 trained | neighbors used: 51864
Anchor 18/20 trained | neighbors used: 52292
Anchor 19/20 trained | neighbors used: 54400
Anchor 20/20 trained | neighbors used: 50989
Training time: 6.6812 s

================

,avg_cpu_util_percent,peak_cpu_util_percent,avg_ram_mb,peak_ram_mb,avg_gpu_util_percent,peak_gpu_util_percent,avg_gpu_memory_util_percent,peak_gpu_memory_util_percent,avg_vram_mb,peak_vram_mb,avg_gpu_power_w,peak_gpu_power_w,avg_gpu_temperature_c,peak_gpu_temperature_c,gpu_energy_joules,run,training_time_s,Algorithm,Device
0,41.271,52.400,213.352,214.422,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,6.681,LLORMA,CPU
1,43.497,52.400,217.262,219.203,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,9.345,LLORMA,CPU
2,38.730,51.700,219.215,219.289,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,9.929,LLORMA,CPU
3,47.679,52.400,219.314,219.484,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,5.762,LLORMA,CPU
4,48.079,52.350,219.486,219.488,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,5.119,LLORMA,CPU
5,43.851,52.250,217.726,218.377,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AVERAGE,7.367,LLORMA,CPU
6,4.048,0.308,2.607,2.215,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STD,2.155,LLORMA,CPU



Saved result file:
/content/drive/MyDrive/LLORMA_Project/hardware_results/LLORMA_CPU_hardware_results.csv


In [ ]:
t0 = time.time()
test_preds = predict_llorma(local_models_cpu, test_u, test_i, global_mean)
cpu_infer_time = time.time() - t0

rmse = np.sqrt(np.mean((test_preds - test_r) ** 2))
mae  = np.mean(np.abs(test_preds - test_r))

print(f"CPU inference time: {cpu_infer_time:.4f}s")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE:  {mae:.4f}")

CPU inference time: 0.0773s
Test RMSE: 1.4531
Test MAE:  1.0978
